# 🧪 W5-D4 概念实验：pass@k、温度、统计显著性与 RTF

> 配套阅读：`ima/第5周-Day4-推理能力评测与Prompt工程.md`（MMLU/GSM8K/HumanEval、RTF 骨架在那边）
>
> 本 notebook 回答四个问题：
> 1. pass@k 的公式 $1-(1-p)^k$ 什么时候会**骗人**？（尝试之间不独立）
> 2. 温度如何同时改变单次准确率与采样多样性？pass@k 有没有**甜点温度**？
> 3. 榜单上差 4 分算领先吗？（bootstrap 置信区间）
> 4. 为什么 RTF（Role-Task-Format）骨架的输出才能被程序消费？

## 实验 1：pass@k 的独立性假设

同一模型、同一题、采样 k 次"至少一次对"。公式假设每次尝试独立同分布。
但真实世界里**题目难度不同**：简单题次次对，难题次次错——尝试之间正相关。
对比三种情形：独立公式 / 独立蒙特卡洛 / 按题难度分层（Beta 分布采样每题真实通过率）。

In [ ]:
import numpy as np

rng = np.random.default_rng(9)
p, n_q, kmax = 0.30, 20_000, 16
ks = [1, 2, 4, 8, 16]

att_ind = rng.random((n_q, kmax)) < p                    # 情形：独立
p_i = rng.beta(2, 5, n_q)                                 # 每题真实通过率（均值≈0.29）
att_cor = rng.random((n_q, kmax)) < p_i[:, None]          # 情形：难度相关

print(f"单次通过率 p≈{p:.0%}（相关情形平均 {p_i.mean():.0%}）")
print(f"{'k':>3} | {'独立公式':>7} | {'独立MC':>7} | {'难度相关MC':>8}")
for k in ks:
    f = 1 - (1 - p) ** k
    a = att_ind[:, :k].any(axis=1).mean()
    b = att_cor[:, :k].any(axis=1).mean()
    print(f"{k:>3} | {f:>7.1%} | {a:>7.1%} | {b:>8.1%}")
print()
print("结论：难度相关时 pass@16 远低于独立公式的 99.5%——")
print("'多试几次总能对'只在简单题上成立，难题的 0 还是 0，分数无法外推到真实任务。")

## 实验 2：温度的权衡——单次准确率 vs 采样覆盖

一半题目里埋一个"陷阱答案"：它的 logit（2.6）比正确答案（2.0）还高——
即模型**最自信的选项是错的**，正确答案排第二但差距不大（真实世界里非常常见）。
温度 T 压缩/放大 logits 后 softmax 采样：
- 单次准确率：T 越低越"自信"，但自信地选错 → 提温才有机会翻盘
- pass@5：5 次采样都挤在同一个答案上（低温）就浪费了配额，太热则乱采

画出两条曲线随 T 的变化。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

rng = np.random.default_rng(4)
n_q, n_wrong, k = 3000, 9, 5
wrong = rng.normal(0.0, 1.0, (n_q, n_wrong))
trap = rng.random(n_q) < 0.5                       # 一半题有"更诱人的错误答案"
wrong[:, 0] = np.where(trap, 2.6, wrong[:, 0])     # 陷阱答案 logit 高于正确答案
logits = np.concatenate([wrong, 2.0 + np.zeros((n_q, 1))], axis=1)  # 正确答案在最后

def softmax_T(z, T):
    z = z / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

Ts = [0.3, 0.5, 0.7, 1.0, 1.4, 2.0, 3.0]
acc1_list, passk_list, uniq_list = [], [], []
for T in Ts:
    probs = softmax_T(logits, T)
    acc1_list.append(probs[:, -1].mean())          # 单次采样期望正确率
    cum = np.cumsum(probs, axis=1)
    u = rng.random((n_q, k))                       # k 次独立采样（逆 CDF）
    draws = (u[:, :, None] > cum[:, None, :]).sum(axis=2).clip(0, n_wrong)
    passk_list.append((draws == n_wrong).any(axis=1).mean())
    uniq_list.append(np.mean([len(set(d)) for d in draws]))

plt.figure(figsize=(7, 4))
plt.plot(Ts, [a * 100 for a in acc1_list], "s-", label="单次采样准确率")
plt.plot(Ts, [a * 100 for a in passk_list], "o-", label=f"pass@{k}（{k} 次采样至少一次对）")
plt.xlabel("温度 T"); plt.ylabel("正确率 (%)")
plt.title("温度权衡：低温自信地错，高温胡乱采，pass@k 有甜点区")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

best_i = int(np.argmax(passk_list))
print(f"pass@{k} 甜点温度 ≈ T={Ts[best_i]}（{passk_list[best_i]:.1%}），")
print(f"该温度下 5 次采样平均覆盖 {uniq_list[best_i]:.1f} 个不同答案（低温只有 {uniq_list[0]:.1f} 个）。")
print("工程含义：模型存在系统性偏见（陷阱答案）时，低温=反复犯同一个错；")
print("走'采样+筛选'路线（自洽性/验证重试）应调到甜点温度，要一次答对则降温。")

## 实验 3：差 4 分算领先吗？——评测样本量与置信区间

模型 A 真实准确率 62%，B 是 58%。用 n 道题各评一次，对准确率差做 bootstrap。
看 n=20 / 50 / 200 时 95% 置信区间：区间盖住 0 就**不能**宣布 A 更强。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

rng = np.random.default_rng(6)
pA, pB = 0.62, 0.58

plt.figure(figsize=(7, 4))
for i, n in enumerate([20, 50, 200]):
    A = rng.random((2000, n)) < pA
    B = rng.random((2000, n)) < pB
    d = A.mean(axis=1) - B.mean(axis=1)
    lo, hi = np.percentile(d, [2.5, 97.5])
    mean = d.mean()
    plt.errorbar(i, mean, yerr=[[mean - lo], [hi - mean]], fmt="o", capsize=6,
                 label=f"n={n:<4} CI [{lo:+.3f}, {hi:+.3f}]")
plt.axhline(0, color="red", ls="--", lw=1)
plt.xticks(range(3), ["n=20", "n=50", "n=200"])
plt.ylabel("准确率差 A−B（真实 +0.04）")
plt.title("样本太少时，4 分领先可能盖住 0 —— 差异不可信")
plt.legend(fontsize=8); plt.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print("n=20/50 的区间大概率跨过 0 → 'A 比 B 好'这时的结论只是噪声；")
print("n=200 才稳定落在 0 右侧。评测预算不够时，先加题目数，再谈Prompt谁强谁弱。")

## 实验 4：RTF 骨架——让输出"可被程序消费"

RTF = Role(角色) + Task(任务/成功标准) + Format(输出格式)。
用纯 Python 模拟两种回答流向下游解析器的结局：结构化 JSON vs 自由发挥散文。

In [ ]:
import json

def rtf_prompt(role, task, fmt):
    return f"角色：{role}\n任务：{task}\n格式：{fmt}"

prompt = rtf_prompt(
    "零售经营分析师",
    "根据日销售额与成本找出异常并给出两条建议；数据不足时明确说明缺什么",
    "输出 JSON，字段 summary / anomalies / actions / missing_data",
)

simulated = {
    "RTF 结构化回答": '{"summary": "周三销售额异常偏低", "anomalies": ["周三环比 -38%"],'
                  ' "actions": ["核对当日促销配置", "检查缺货SKU"], "missing_data": []}',
    "自由发挥回答": "我觉得周三卖得不太好，可能是天气原因吧，建议多搞点活动吸引顾客～",
}

print("=== Prompt ===")
print(prompt)
print()
for name, resp in simulated.items():
    try:
        obj = json.loads(resp)
        status = f"解析成功 → 字段 {list(obj)}"
    except json.JSONDecodeError:
        status = "解析失败 → 下游程序拿不到任何字段，只能人工读"
    print(f"[{name}] {status}")
print()
print("Format 的价值不是'好看'：它把 LLM 输出变成流水线的输入，也把'缺数据'变成可检测的信号。")

## 结论

- pass@k 公式隐含**独立性假设**，题目难度相关时高估（实验 1）
- 温度是"单次准确率 ↔ 多样性"的旋钮：采样+筛选路线有甜点温度（实验 2）
- 小样本榜单差几分毫无意义，先算置信区间（实验 3）
- RTF 的 Format 让输出可解析、缺口可检测（实验 4）

→ 深入阅读：`ima/第5周-Day4-推理能力评测与Prompt工程.md`（三类基准细节、数据污染、策略阶梯）